# Practice 33 — pandas + NumPy
Week 4 consolidation — no new concepts. Full review of everything learned so far.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Task 1 — Week 4 in one place

Load `sns.load_dataset('tips')`. This task covers every Week 4 concept without stopping.

1. Record memory before any changes (`deep=True`). Convert `sex`, `smoker`, `day`, `time` to `category`. How much did memory shrink?
2. Define an ordered dtype for `day` with a meaningful order: `Thur < Fri < Sat < Sun`. Apply it. Filter to only weekend days (Sat and Sun) using the ordered comparison — no `.isin()`.
3. On the filtered result, call `.cat.remove_unused_categories()` on `day`. Confirm only `'Sat'` and `'Sun'` remain.
4. Print `.cat.codes` for the first 5 rows of `day` (full DataFrame). Build a dict mapping each day name to its code using a dict comprehension.
5. `groupby('day', observed=True)['tip'].mean()` — which day has the highest average tip?
6. `pivot_table` of mean tip by `day` × `smoker`, `observed=True`. Use `.loc` to extract the `'Sat'` row.

In [ ]:
# Your code here
tips = sns.load_dataset('tips')
#print('original: ', tips.memory_usage(deep = True))
#print(tips.dtypes)
ou = tips.memory_usage(deep = True).sum()
tips[['sex','smoker','day','time']] = tips[['sex','smoker','day','time']].astype('category')
#print('new: ',tips.memory_usage(deep=True))
nu = tips.memory_usage(deep = True).sum()
print('shrinkage: ', ou - nu)
### looks like the original is already in category datatype 
do = pd.CategoricalDtype(categories=['Thur','Fri','Sat','Sun'], ordered=True)
tips['day'] = tips['day'].astype(do)

f = tips[tips['day']>'Fri'].copy()
f['day'] = f['day'].cat.remove_unused_categories()
f['day'].cat.categories


print(tips['day'][0:5].cat.codes)

d = {v:k for k,v in enumerate(tips['day'].cat.categories)}
d

tips.groupby('day',observed=True)['tip'].mean().idxmax()


p = tips.pivot_table(
    index = 'day',
    columns='smoker',
    values='tip',
    observed=True
)
p.loc['Sat']

shrinkage:  0
0    3
1    3
2    3
3    3
4    3
dtype: int8


smoker
Yes    2.875476
No     3.102889
Name: Sat, dtype: float64

---
## Task 2 — New dataset, open exploration

Load `sns.load_dataset('flights')`. This dataset has three columns: `year` (1949–1960), `month` (Jan–Dec as strings), and `passengers`.

No structure. Explore it however you want, but answer these:

1. Convert `month` to an ordered category in calendar order (`Jan < Feb < ... < Dec`). Which month consistently has the most passengers? Use whatever method you prefer.
2. Pivot the data so rows are years and columns are months — this gives a 12×12 table of passenger counts. Which year had the highest total passengers?
3. Stack the pivot table into a long Series with a `(year, month)` MultiIndex. Use `.xs()` to pull out all rows for `'Jul'`.
4. Among summer months (Jun, Jul, Aug), what is the mean passenger count per year? Use NumPy.

In [108]:
# Your code here
flights = sns.load_dataset('flights')

#1
mo = pd.CategoricalDtype(categories=list(flights.month.cat.categories),
                         ordered=True)
flights['month'] = flights['month'].astype(mo)
ms = flights.groupby('month',observed=True)['passengers'].min()
max_ms = ms.max()
s1 = ms.index[ms==max_ms]

avs = flights.groupby('month',observed=True)['passengers'].mean()
avs = avs[s1]
max_avs = avs.max()
s2 = avs.index[avs==max_avs]

answer = set(s1) & set(s2)
answer

#2
p = flights.pivot_table(
    index = 'year',
    columns='month',
    observed=True,
    aggfunc='sum'
)
p.sum(axis=1).idxmax()

#3
s = p.stack()
s.xs('Jul', level='month')

#4
summer_flight = flights[flights['month'].isin(['Jun','Jul','Aug'])]
summer_flight.groupby('month',observed=True)['passengers'].agg('mean')


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_35765/2517483522.py:30: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  s = p.stack()


month
Jun    311.666667
Jul    351.333333
Aug    351.083333
Name: passengers, dtype: float64

---
## Task 3 — Reshape challenge

Load `sns.load_dataset('titanic')`. No `.pipe()` in this task.

Your goal: produce a clean summary of **median fare** by `pclass` × `embarked`, sliceable by either level.

Requirements — in whatever order makes sense to you:
- `pclass` must be an ordered category (`3 < 2 < 1`), `embarked` unordered category
- The final reshaped object must be a Series with a `(pclass, embarked)` MultiIndex
- Use `.xs()` with a tuple to extract one specific cell
- Use `.loc` with an ordered comparison to filter the Series to only first and second class rows
- Find the `(pclass, embarked)` combination with the highest median fare — without `.idxmax()`

In [ ]:
# Your code here

titanic = sns.load_dataset('titanic')

co = pd.CategoricalDtype(categories=[3,2,1], ordered=True)
titanic['pclass'] = titanic['pclass'].astype(co)
titanic['embarked'] = titanic['embarked'].astype('category')

ss = titanic.pivot_table(
    index='pclass',
    columns='embarked',
    observed=True,
    values = 'fare',
    aggfunc='median'
)

ss_final = ss.stack()

ss_final.xs((3,'C'))

hc = titanic.pclass[titanic.pclass>3].unique()
ss_final.loc[hc]


ss_final.index[np.argmax(ss_final)]

ss_final



pclass  embarked
3       C            7.8958
        Q            7.7500
        S            8.0500
2       C           24.0000
        Q           12.3500
        S           13.5000
1       C           78.2667
        Q           90.0000
        S           52.0000
dtype: float64

---
## Task 4 — Pipe

Load `sns.load_dataset('diamonds')`. Write a `.pipe()` chain — 3 functions, all written by you:

- One sets up ordered `cut` and `color` category dtypes and checks how much memory the conversion saved (print it inside the function)
- One adds `price_per_carat` and a `color_avg_ppc` column (mean `price_per_carat` per color group using `transform`)
- One filters to diamonds where `price_per_carat` is above their color group average

After the chain:
- Use `.agg()` with a dict of tuples: mean `price`, median `price_per_carat`, count — per `cut`. Use `observed=True`.
- Which cut has the most above-average-value diamonds? Use `np.argmax()`.
- Use `.loc` to extract only `'Ideal'` cut diamonds from the result. What is the mean `carat` for this group?

In [107]:
# Your code here

diamonds =sns.load_dataset('diamonds')

def catit(df):
    om = df.memory_usage(deep = True).sum()
    df[['cut','color']] = df[['cut','color']].astype('category')
    nm = df.memory_usage(deep = True).sum()
    print('shrinkage: ', om-nm)
    return df

def add_p(df):
    df['price_per_carat'] = df['price']/df['carat']
    df['color_avg_ppc'] = df.groupby('color', observed = True)['price_per_carat'].transform('mean')
    return df

def fil(df):
    fil = df[df['price_per_carat']> df['color_avg_ppc']].copy()
    return fil 

res = diamonds.copy().pipe(catit).pipe(add_p).pipe(fil)

ss = res.groupby('cut', observed=True).agg(
    mean_price = ('price','mean'),
    median_ppc = ('price_per_carat', 'median'),
    count = ('price','size')
)
print(ss.index[np.argmax(ss['median_ppc'])])

res.loc[res['cut'] == 'Ideal','carat'].mean()


shrinkage:  0
Premium


np.float64(1.141149942181678)